In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import string
import os


captcha_len = 4
characters = string.digits + string.ascii_uppercase
num_classes = len(characters)

height = 40
width = 95
learning_rate = 0.001
epochs = 50
batch_size = 64

char_to_idx = {c: i for i, c in enumerate(characters)}
idx_to_char = {i: c for i, c in enumerate(characters)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

class Captchadataset(Dataset) :
    def __init__(self, data_dir = None, images = None, labels = None):
        if data_dir is not None :
            self.filepaths = []
            self.labels = []
            for filename in os.listdir(data_dir) :
                if filename.endswith('.npy') :
                    label = os.path.splitext(filename)[0]
                    if len(label) != captcha_len :
                        continue
                    filepath = os.path.join(data_dir, filename)
                    self.filepaths.append(filepath)
                    self.labels.append(label)
                else :
                    print("Unexpected file found\n")
            print(f"Loaded {len(self.labels)} npy files")
        else :
            print("Files not found")
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        if hasattr(self, 'filepaths') and self.filepaths is not None:
            image = np.load(self.filepaths[idx]).astype(np.float32)
        else :
            image = self.images[idx]
        image = torch.FloatTensor(image).unsqueeze(0)

        label_str = self.labels[idx]
        label = torch.LongTensor([char_to_idx[c] for c in label_str])

        return image, label
        

def prepare_data_from_folder():
    data_dir = './data/'
    full_dataset = Captchadataset(data_dir = data_dir)

    total = len(full_dataset)
    train_size = int(0.8 * total)
    test_size = total - train_size

    train_dataset, test_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, test_size]
    )
    return train_dataset, test_dataset

train_dataset, test_dataset = prepare_data_from_folder()

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = True)

print(f"Training set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")

class CaptchaCNN(nn.Module):
    def __init__(self) :
        super(CaptchaCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels = 1,
                out_channels = 32,
                kernel_size = 3,
                padding = 1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace = True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), 
            nn.BatchNorm2d(64),
            nn.ReLU(inplace = True), 
            nn.MaxPool2d(2, 2), 

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), 
            nn.BatchNorm2d(128),
            nn.ReLU(inplace = True), 
            nn.MaxPool2d(2, 2), 
        )

        self._feature_size = self._get_feature_size()

        self.fc_shared = nn.Sequential(
            nn.Linear(self._feature_size, 512), 
            nn.ReLU(inplace = True), 
            nn.Dropout(0.2)
        )

        self.fc_heads = nn.ModuleList([
            nn.Linear(512, num_classes) for _ in range(captcha_len)
        ])

    def _get_feature_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, height, width)
            out = self.features(dummy)
            return out.view(1, -1).size(1)
    
    def forward(self, x) :

        x = self.features(x)

        x = x.view(x.size(0), -1)

        x = self.fc_shared(x)

        outputs = [head(x) for head in self.fc_heads]

        return outputs

model = CaptchaCNN().to(DEVICE)
print(model)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)


def train_one_epoch(model, train_loader, criterion, optimizer, device) :
    model.train() #Switch to training mode, dropout activated, batchNorm updating

    total_loss = 0
    correct_chars = 0
    correct_captchas = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device) #Move the images and labels to the GPU
        labels = labels.to(device)

        outputs = model(images)
        loss = 0
        for i in range(captcha_len) :
            loss += criterion(outputs[i], labels[:, i])

        optimizer.zero_grad()
        loss.backward()     
        optimizer.step()      

        #Calculate accuracy

        total_loss += loss.item()
        batch_size = images.size(0)
        total_samples += batch_size

        preds = torch.stack([out.argmax(dim = 1) for out in outputs], dim = 1)
        #Single Character Accuracy
        correct_chars += (preds == labels).sum().item()
        #captcha Accuracy
        correct_captchas += (preds == labels).all(dim = 1).sum().item()

    avg_loss = total_loss / len(train_loader)
    char_acc = correct_chars / (total_samples * captcha_len) * 100
    captcha_acc = correct_captchas / total_samples * 100
    
    return avg_loss, char_acc, captcha_acc


def evaluate(model, data_loader, criterion, device) :
    model.eval() #Switch to evaluation mode

    total_loss = 0
    correct_chars = 0
    correct_captchas = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device) 
            labels = labels.to(device)

            outputs = model(images)
            loss = 0
            for i in range(captcha_len) :
                loss += criterion(outputs[i], labels[:, i])

            #optimizer.zero_grad()
            #loss.backward() 
            #optimizer.step() 

            total_loss += loss.item()
            batch_size = images.size(0)
            total_samples += batch_size

            preds = torch.stack([out.argmax(dim = 1) for out in outputs], dim = 1)

            correct_chars += (preds == labels).sum().item()
    
            correct_captchas += (preds == labels).all(dim = 1).sum().item()

    avg_loss = total_loss / len(data_loader)
    char_acc = correct_chars / (total_samples * captcha_len) * 100
    captcha_acc = correct_captchas / total_samples * 100
    
    return avg_loss, char_acc, captcha_acc

def train():
    best_captcha_acc = 0

    print("=" * 70)
    print(f"Start training | Epochs: {epochs} | Batch Size: {batch_size}")
    print(f"Character set size: {num_classes} | Captcha length: {captcha_len}")
    print("=" * 70)

    for epoch in range(1, epochs + 1):
        train_loss, train_char_acc, train_cap_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE
        )

        test_loss, test_char_acc, test_cap_acc = evaluate(
            model, test_loader, criterion, DEVICE
        )

        print(f"Epoch [{epoch:3d}/{epochs}] | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Char Acc: {train_char_acc:.2f}% | "
              f"Train Cap Acc: {train_cap_acc:.2f}% | "
              f"Test Char Acc: {test_char_acc:.2f}% | "
              f"Test Cap Acc: {test_cap_acc:.2f}%")
        if test_cap_acc > best_captcha_acc :
            best_captcha_acc = test_cap_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_acc': best_captcha_acc,
            }, 'best_captcha_model.pth')
            print(f"  * Best model saved! Test set overall accuracy: {best_captcha_acc:.2f}%")
    
    
    print("=" * 70)
    print(f"Training complete! Best test set overall accuracy: {best_captcha_acc:.2f}%")

train()